## Ejercicio 4 - Simulación de ventas de Don Francisco y Don Miguel

**Consigna:**

a) Simular el número de clientes diarios de Don Francisco y de Don Miguel entre 2023 y 2025, usando Poisson con λ_t = efecto anual + efecto mensual + efecto diario + efecto local.

b) ECDF y densidad aproximada de las ventas de Don Francisco, por año.

c) ECDF y densidad aproximada de las ventas de Don Miguel, por año.

d) Densidad empírica conjunta de ambos almacenes.

Se toma el número de clientes diarios simulado como la variable "ventas" a analizar en b), c) y d), ya que es el único dato generado por el modelo (no hay un monto en dólares definido en la consigna).

### Librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
np.random.seed(42)  # reproducibilidad


### a) Simulación del número de clientes diarios

Para cada fecha y cada propietario, $\lambda_t$ es la suma de los cuatro efectos de la tabla del enunciado.
`fecha.weekday()` de pandas devuelve 0=Lunes, ..., 6=Domingo, por lo que se arma el diccionario `efecto_diario` con esa convención.

In [ ]:
# Efecto anual
efecto_anual = {2023: 10, 2024: 15, 2025: 20}

# Efecto mensual (1=Enero, ..., 12=Diciembre)
efecto_mensual = {
    1: 10, 2: 15, 3: 20, 4: 20, 5: 25, 6: 25,
    7: 30, 8: 25, 9: 25, 10: 20, 11: 15, 12: 10,
}

# Efecto diario. pandas: weekday() -> 0=Lunes ... 6=Domingo
efecto_diario = {
    0: 20,  # Lunes
    1: 30,  # Martes
    2: 35,  # Miércoles
    3: 30,  # Jueves
    4: 20,  # Viernes
    5: 10,  # Sábado
    6: 10,  # Domingo
}

# Efecto local (propietario)
efecto_local = {"Don Francisco": 20, "Don Miguel": 10}


In [ ]:
def lam(fecha, propietario):
    return (
        efecto_anual[fecha.year]
        + efecto_mensual[fecha.month]
        + efecto_diario[fecha.weekday()]
        + efecto_local[propietario]
    )

fechas = pd.date_range("2023-01-01", "2025-12-31", freq="D")

registros = []
for fecha in fechas:
    for propietario in ("Don Francisco", "Don Miguel"):
        lam_t = lam(fecha, propietario)
        clientes = np.random.poisson(lam_t)
        registros.append({
            "fecha": fecha,
            "anio": fecha.year,
            "propietario": propietario,
            "lambda": lam_t,
            "clientes": clientes,
        })

df = pd.DataFrame(registros)
df.head()


In [ ]:
# Formato ancho: una columna de clientes por propietario, alineadas por fecha.
# Se usa en el punto d) para la densidad conjunta.
df_wide = df.pivot(index="fecha", columns="propietario", values="clientes").reset_index()
df_wide.columns.name = None
df_wide.head()


### b) ECDF y densidad aproximada de las ventas de Don Francisco, por año

In [ ]:
df_francisco = df[df["propietario"] == "Don Francisco"]

fig, axs = plt.subplots(2, 3, figsize=(15, 8))

for i, anio in enumerate([2023, 2024, 2025]):
    data_anio = df_francisco.loc[df_francisco["anio"] == anio, "clientes"]

    # Función empírica de distribución (ECDF)
    sns.ecdfplot(data_anio, ax=axs[0, i], color="red")
    axs[0, i].set_title(f"ECDF Don Francisco {anio}")
    axs[0, i].set_xlabel("Clientes diarios (ventas)")
    axs[0, i].set_ylabel("Proporción acumulada")

    # Aproximación a la densidad: histograma normalizado + KDE
    sns.histplot(data_anio, stat="density", discrete=True, color="skyblue", ax=axs[1, i])
    sns.kdeplot(data_anio, color="red", ax=axs[1, i])
    axs[1, i].set_title(f"Densidad Don Francisco {anio}")
    axs[1, i].set_xlabel("Clientes diarios (ventas)")
    axs[1, i].set_ylabel("Densidad")

plt.tight_layout()
plt.show()


### c) ECDF y densidad aproximada de las ventas de Don Miguel, por año

In [ ]:
df_miguel = df[df["propietario"] == "Don Miguel"]

fig, axs = plt.subplots(2, 3, figsize=(15, 8))

for i, anio in enumerate([2023, 2024, 2025]):
    data_anio = df_miguel.loc[df_miguel["anio"] == anio, "clientes"]

    sns.ecdfplot(data_anio, ax=axs[0, i], color="red")
    axs[0, i].set_title(f"ECDF Don Miguel {anio}")
    axs[0, i].set_xlabel("Clientes diarios (ventas)")
    axs[0, i].set_ylabel("Proporción acumulada")

    sns.histplot(data_anio, stat="density", discrete=True, color="lightgreen", ax=axs[1, i])
    sns.kdeplot(data_anio, color="red", ax=axs[1, i])
    axs[1, i].set_title(f"Densidad Don Miguel {anio}")
    axs[1, i].set_xlabel("Clientes diarios (ventas)")
    axs[1, i].set_ylabel("Densidad")

plt.tight_layout()
plt.show()


### d) Densidad empírica conjunta de ambos almacenes

Se usa `sns.jointplot(kind="kde")`, igual que en el notebook de referencia, tomando para cada fecha el par (clientes de Don Francisco, clientes de Don Miguel). El mapa de calor central es la densidad conjunta y las curvas de los márgenes son las densidades marginales de cada comerciante.

In [ ]:
sns.jointplot(
    data=df_wide,
    x="Don Francisco",
    y="Don Miguel",
    kind="kde",
    fill=True,
    cmap="Blues",
)
plt.suptitle("Densidad empírica conjunta - Don Francisco vs Don Miguel (2023-2025)", y=1.02)
plt.show()


Opcionalmente, se puede repetir la densidad conjunta discriminando por año para ver si la relación entre ambos comercios cambia con el tiempo (por ejemplo, si comparten el mismo flujo de clientes del barrio).

In [ ]:
g = sns.jointplot(
    data=df_wide.assign(anio=df_wide["fecha"].dt.year),
    x="Don Francisco",
    y="Don Miguel",
    hue="anio",
    kind="scatter",
    alpha=0.5,
)
plt.suptitle("Clientes diarios: Don Francisco vs Don Miguel, por año", y=1.02)
plt.show()
